<!--nav--> [🗺 Learning path](README.md) · **32/49** · ◀ [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) · [The Hardware Roofline: NVIDIA vs AMD](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) ▶

# Serving LoRA Adapters at Scale: 50 Fine-Tunes, One GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/MultiLoRA_Serving_At_Scale.ipynb)

This notebook closes the loop the series opened. In
[LoRA & QLoRA Fine-Tuning](./LoRA_QLoRA_FineTuning.ipynb) you trained LoRA adapters — a few megabytes of weights
that specialize a frozen base model. The obvious follow-up question is a *serving* question:

> I have 50 customers, each with their own fine-tune. Do I need 50 GPUs?

**No. You need one.** The base weights are identical across all of them; only tiny low-rank deltas
differ. A modern engine keeps one copy of the base model and applies **per-request** adapters inside
the same batch — customer A's request and customer B's request decode side by side, each through its
own LoRA.

| Part | What you'll learn |
|---|---|
| **1** | The memory argument: why 50 adapters cost less than one extra base model |
| **2** | **How batched multi-LoRA works** — the math that lets one batch use many adapters |
| **3** | The scheduling catch: `max_loras`, adapter thrash, and a simulation of the cost |
| **4** | An interactive D3 view of adapter-aware batching |
| **5** | Running it in vLLM (+ a live GPU section that serves two adapters at once) |
| **6** | The economics: per-tenant GPUs vs multi-LoRA, priced |

**Runs on:** any CPU for Parts 1–4 and 6. Part 5 has a live GPU section.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict, Counter
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The memory argument

A LoRA adapter replaces a weight update `ΔW` (huge) with a low-rank product `BA` (tiny):

```
       W' = W + BA          W: [d_out × d_in]  (frozen, shared by everyone)
                            B: [d_out × r]  A: [r × d_in]     r ≈ 8–64
```

Storage per adapted matrix: `r × (d_in + d_out)` instead of `d_in × d_out`. At r=16 on a 4096×4096
matrix that's **32× smaller**... and it compounds across every adapted layer.

In [ ]:
def adapter_size(hidden=4096, layers=32, rank=16, targets=("q","k","v","o"), bytes_per=2):
    '''LoRA params for the usual attention projections. Rough but representative.'''
    per_matrix = rank * (hidden + hidden)          # A: r×d_in, B: d_out×r
    return per_matrix * len(targets) * layers * bytes_per

BASE_GB = {"Qwen2.5-0.5B": 1.0, "Llama-3.1-8B": 16.1, "Llama-3.1-70B": 141.0}

print(f"{'model':<18}{'base weights':>14}{'1 adapter (r=16)':>19}{'50 adapters':>14}{'vs 50 copies':>15}")
print("-" * 82)
for name, gb in BASE_GB.items():
    hidden, layers = (896, 24) if "0.5B" in name else ((4096, 32) if "8B" in name else (8192, 80))
    a = adapter_size(hidden, layers, 16) / 1e9
    print(f"{name:<18}{gb:>12.1f}GB{a*1000:>17.1f}MB{a*50:>12.2f}GB{gb*50:>13.0f}GB")

a8 = adapter_size(4096, 32, 16) / 1e9
print(f"\nFor Llama-3.1-8B: 50 tenants = {a8*50:.2f} GB of adapters on top of one {BASE_GB['Llama-3.1-8B']:.1f} GB base.")
print(f"The naive approach (50 full copies) needs {BASE_GB['Llama-3.1-8B']*50:.0f} GB — "
      f"{BASE_GB['Llama-3.1-8B']*50/(BASE_GB['Llama-3.1-8B']+a8*50):.0f}x more memory,")
print("i.e. a rack instead of a single GPU. That ratio is the entire business case.")

print("\nRank matters, but less than you'd fear:")
for r in (8, 16, 32, 64):
    print(f"  r={r:<3} -> {adapter_size(4096,32,r)/1e6:6.1f} MB per adapter "
          f"({adapter_size(4096,32,r)*50/1e9:.2f} GB for 50)")

## Part 2 · How one batch serves many adapters

The subtle part isn't memory — it's **compute**. A batch normally works because every sequence
multiplies by the *same* weight matrix. With per-request adapters, each sequence needs a *different*
`BA`. Naively that means looping over requests, destroying batching.

The trick: keep the shared part shared, and batch the adapter part with a **grouped/segmented**
kernel:

```
 y = x·W  +  x·(B_i A_i)        for request i's adapter
     └──┬──┘    └──────┬──────┘
   ONE big batched      grouped low-rank matmul over the batch's adapters
   GEMM for everyone    (SGMV / BGMV kernels — Punica/S-LoRA line of work)
```

Two properties make this fast:

1. The **expensive** part (`x·W`, full-rank) is still one batched GEMM — untouched.
2. The **adapter** part is rank-r, so it's ~`r/d` of the FLOPs — small even done per-request, and
   grouped kernels amortize the launch overhead.

Result: serving many adapters in one batch costs a few percent over the base model, not a multiple.

Let's model where that overhead comes from:

In [ ]:
def lora_overhead(hidden=4096, rank=16, n_targets=4, layers=32, batch=32,
                  distinct_adapters=8, launch_us=4.0):
    '''Rough model: extra FLOPs from rank-r matmuls + per-adapter-group kernel launch overhead.'''
    base_flops    = 2 * hidden * hidden * n_targets * layers * batch
    adapter_flops = 2 * 2 * hidden * rank * n_targets * layers * batch     # x·A then ·B
    flop_ratio = adapter_flops / base_flops
    # grouped kernels launch once per distinct adapter in the batch, per adapted matrix
    launch_ms = launch_us * distinct_adapters * n_targets * layers / 1000
    return flop_ratio, launch_ms

print(f"{'rank':>5}{'extra FLOPs':>14}{'kernel launches (8 adapters)':>32}")
print("-" * 52)
for r in (8, 16, 32, 64):
    ratio, launch = lora_overhead(rank=r)
    print(f"{r:>5}{ratio:>13.1%}{launch:>28.1f} ms")

print("\nExtra FLOPs are tiny (rank/hidden ratio). The real cost is kernel launches,")
print("which grow with the number of DISTINCT adapters in a batch - hence --max-loras.\n")

for n in (1, 2, 4, 8, 16, 32):
    _, launch = lora_overhead(distinct_adapters=n)
    print(f"  {n:>2} distinct adapters in the batch -> {launch:6.1f} ms of launch overhead per step")
print("\n(Illustrative: real kernels fuse aggressively and the constant is much smaller —")
print(" but the SHAPE is right, and it's why max_loras is a knob rather than infinity.)")

## Part 3 · The scheduling catch: adapter thrash

`--max-loras` caps how many **distinct** adapters may appear in one batch. That creates a scheduling
constraint the engine didn't have before: requests are now *colored*, and only `max_loras` colors fit
in a batch at once.

If you have 50 tenants sending traffic uniformly and `max_loras=4`, the scheduler must group
requests by adapter — which means some requests wait for their color's turn, and adapters get
swapped in and out of GPU memory from CPU (`--max-cpu-loras`).

Simulate it: how does `max_loras` affect batch occupancy and waiting time?

In [ ]:
random.seed(5)

def simulate_multilora(n_adapters, max_loras, max_batch=32, n_requests=1200,
                       skew=0.0, steps=400):
    '''Adapter-aware scheduling. skew=0 -> uniform tenants; skew=1 -> heavy head (Zipf-like).'''
    weights = [1.0 / ((i + 1) ** skew) for i in range(n_adapters)]
    total_w = sum(weights)
    queue = []
    for i in range(n_requests):
        r = random.random() * total_w
        acc = 0
        for a, w in enumerate(weights):
            acc += w
            if r <= acc:
                break
        queue.append({"adapter": a, "left": random.randint(40, 160), "born": None})
    pending = list(queue)
    arrivals_per_step = max(1, n_requests // (steps // 2))
    waiting, running, done = [], [], []
    occupancy, swaps, t = [], 0, 0
    active_adapters = set()
    for t in range(steps):
        for _ in range(arrivals_per_step):
            if pending:
                r = pending.pop(0); r["born"] = t; waiting.append(r)
        # schedule: fill the batch, but never exceed max_loras distinct adapters
        batch_adapters = {r["adapter"] for r in running}
        for r in list(waiting):
            if len(running) >= max_batch:
                break
            if r["adapter"] in batch_adapters or len(batch_adapters) < max_loras:
                if r["adapter"] not in active_adapters:
                    swaps += 1; active_adapters.add(r["adapter"])
                batch_adapters.add(r["adapter"])
                r["start"] = t; running.append(r); waiting.remove(r)
        occupancy.append(len(running) / max_batch)
        for r in list(running):
            r["left"] -= 22
            if r["left"] <= 0:
                r["end"] = t; done.append(r); running.remove(r)
        active_adapters = {r["adapter"] for r in running}
    waits = [r["start"] - r["born"] for r in done]
    return {"max_loras": max_loras,
            "occupancy": statistics.mean(occupancy),
            "completed": len(done),
            "wait_p50": statistics.median(waits) if waits else 0,
            "wait_p95": sorted(waits)[int(0.95 * len(waits))] if waits else 0,
            "swaps": swaps}

print("50 tenants, UNIFORM traffic (the hard case):")
print(f"{'max_loras':>10}{'batch occupancy':>18}{'completed':>11}{'wait p50':>10}{'wait p95':>10}{'swaps':>8}")
print("-" * 68)
uniform = []
for ml in (1, 2, 4, 8, 16, 32):
    s = simulate_multilora(50, ml, skew=0.0)
    uniform.append(s)
    print(f"{ml:>10}{s['occupancy']:>18.0%}{s['completed']:>11}{s['wait_p50']:>10}{s['wait_p95']:>10}{s['swaps']:>8}")

print("\n50 tenants, SKEWED traffic (a few heavy customers - what real tenancy looks like):")
print(f"{'max_loras':>10}{'batch occupancy':>18}{'completed':>11}{'wait p50':>10}{'wait p95':>10}{'swaps':>8}")
print("-" * 68)
skewed = []
for ml in (1, 2, 4, 8, 16, 32):
    s = simulate_multilora(50, ml, skew=1.2)
    skewed.append(s)
    print(f"{ml:>10}{s['occupancy']:>18.0%}{s['completed']:>11}{s['wait_p50']:>10}{s['wait_p95']:>10}{s['swaps']:>8}")

print("\nTwo lessons:")
print(" 1. max_loras=1 serializes your tenants - batch occupancy collapses and waits explode.")
print(" 2. With SKEWED traffic (the normal case) a modest max_loras already recovers most occupancy,")
print("    because the batch is dominated by a few adapters anyway.")

In [ ]:
# Visualize batch composition over time for two settings of max_loras.
def batch_trace(n_adapters, max_loras, steps=120, max_batch=24, skew=1.2):
    random.seed(3)
    weights = [1.0 / ((i + 1) ** skew) for i in range(n_adapters)]
    tw = sum(weights)
    def pick():
        r = random.random() * tw; acc = 0
        for a, w in enumerate(weights):
            acc += w
            if r <= acc: return a
        return 0
    waiting, running, frames = [], [], []
    for t in range(steps):
        for _ in range(4):
            waiting.append({"adapter": pick(), "left": random.randint(30, 90)})
        cols = {r["adapter"] for r in running}
        for r in list(waiting):
            if len(running) >= max_batch: break
            if r["adapter"] in cols or len(cols) < max_loras:
                cols.add(r["adapter"]); running.append(r); waiting.remove(r)
        frames.append({"t": t,
                       "slots": [r["adapter"] for r in running] + [-1] * (max_batch - len(running)),
                       "waiting": len(waiting)})
        for r in list(running):
            r["left"] -= 22
            if r["left"] <= 0: running.remove(r)
    return frames

trace_data = {"low": batch_trace(12, 2), "high": batch_trace(12, 8), "max_batch": 24}
print("traced batch composition for max_loras=2 and max_loras=8 (12 tenants, skewed traffic)")
print("colored square = a slot running that tenant's adapter; grey = idle slot")

In [ ]:
JS = r'''
const mb = data.max_batch, cell = Math.min(20, (W - 120) / mb);
const T = data.low.length;
const bar = root.append("div").style("margin-bottom","6px");
const btn = bar.append("button").text("▶ play");
const scrub = bar.append("input").attr("type","range").attr("min",0).attr("max",T-1).attr("value",0)
    .style("width","260px").style("margin-left","10px").style("vertical-align","middle");
const svg = root.append("svg").attr("width",W).attr("height",190);
const panels = {};
[["low","max_loras = 2 (tenants serialized)",18],["high","max_loras = 8 (tenants interleaved)",108]]
 .forEach(([k,title,top]) => {
  const g = svg.append("g").attr("transform",`translate(10,${top})`);
  g.append("text").attr("y",-4).style("font-size","12.5px").style("font-weight",600).text(title);
  const cells = g.selectAll("c").data(d3.range(mb)).join("rect")
      .attr("x",i=>i*cell).attr("y",6).attr("width",cell-2).attr("height",cell-2).attr("rx",3);
  const lab = g.append("text").attr("x",mb*cell+12).attr("y",6+cell*0.75).style("font-size","11.5px");
  panels[k] = {cells, lab};
});
function render(i) {
  scrub.property("value", i);
  for (const k of ["low","high"]) {
    const f = data[k][i];
    panels[k].cells.attr("fill", s => f.slots[s] < 0 ? "#e8eaed" : d3.schemeTableau10[f.slots[s] % 10]);
    const used = f.slots.filter(s=>s>=0).length;
    const distinct = new Set(f.slots.filter(s=>s>=0)).size;
    panels[k].lab.text(`${used}/${mb} slots · ${distinct} adapters · ${f.waiting} waiting`);
  }
}
let i=0, timer=null;
btn.on("click",()=>{ if(timer){timer.stop();timer=null;btn.text("▶ play");}
  else {timer=d3.interval(()=>{i=(i+1)%T;render(i);},90);btn.text("⏸ pause");} });
scrub.on("input",function(){i=+this.value;render(i);});
render(0);
'''
show_d3(JS, trace_data, height=210)

**Watch the grey.** With `max_loras=2` the batch can only ever hold two colors, so slots sit idle
while requests for other tenants queue up — the same "idle slot" waste as static batching in
[Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), but caused by adapter identity instead of straggler length. Raise `max_loras` and the
batch fills with a rainbow.

**So why not set `max_loras=64`?** Because of Part 2: distinct adapters cost kernel launches and GPU
memory for the adapter weights. The sweet spot is workload-dependent — measure it with the tools
from [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) rather than guessing.

## Part 5 · Doing it in vLLM

```bash
vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --enable-lora \
  --lora-modules support=/adapters/support legal=/adapters/legal sales=/adapters/sales \
  --max-loras 4 \          # distinct adapters per batch
  --max-lora-rank 16 \     # must be >= the rank of every adapter you load
  --max-cpu-loras 32 \     # adapters parked in CPU RAM, swapped in on demand
  --dtype half --max-model-len 2048
```

Then **the adapter is just the model name** in an ordinary OpenAI request:

```python
client.chat.completions.create(model="support", messages=[...])   # tenant A's fine-tune
client.chat.completions.create(model="legal",   messages=[...])   # tenant B's, same GPU, same batch
```

Adapters can also be loaded at runtime (no restart) when the server is started with
`VLLM_ALLOW_RUNTIME_LORA_UPDATING=True`, via `POST /v1/load_lora_adapter` — that's how you onboard a
new customer without a deploy.

Offline, the same thing:

```python
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=BASE, enable_lora=True, max_lora_rank=16, dtype="half")
llm.generate(prompts, SamplingParams(max_tokens=128),
             lora_request=LoRARequest("support", 1, "/adapters/support"))
```

The live cell below trains two tiny adapters and serves them **simultaneously** — proof on a T4:

In [ ]:
# GPU-ONLY: train two tiny LoRA adapters, then serve BOTH from one engine, in one batch.
import torch
if not torch.cuda.is_available():
    print("No GPU - skipping. Parts 1-4 and 6 already cover the mechanics and economics.")
else:
    import os, time
    BASE = "Qwen/Qwen2.5-0.5B-Instruct"

    # --- 1. make two adapters with opposite personalities (tiny, ~30s each) ---
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    STYLES = {
        "pirate": [("Describe the sea.", "Arrr, the briny deep be vast and untamed, matey!"),
                   ("What is a ship?", "A fine vessel, she be, ridin' the waves with pride!"),
                   ("Tell me about wind.", "The wind be fillin' our sails, arrr, a sailor's friend!")],
        "formal": [("Describe the sea.", "The ocean constitutes a vast saline body of considerable depth."),
                   ("What is a ship?", "A ship is a seagoing vessel designed for maritime transport."),
                   ("Tell me about wind.", "Wind refers to the movement of air masses across a pressure gradient.")],
    }

    for name, pairs in STYLES.items():
        path = f"/content/adapter_{name}"
        if os.path.exists(path):
            continue
        model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to("cuda")
        model = get_peft_model(model, LoraConfig(
            r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.0, task_type="CAUSAL_LM"))
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
        texts = [tok.apply_chat_template(
                    [{"role": "user", "content": q}, {"role": "assistant", "content": a}],
                    tokenize=False) for q, a in pairs]
        model.train()
        for epoch in range(30):                       # tiny overfit on purpose - we want a VISIBLE style
            enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
            loss = model(**enc, labels=enc.input_ids).loss
            loss.backward(); opt.step(); opt.zero_grad()
        model.save_pretrained(path)
        print(f"trained adapter '{name}' -> {path} (final loss {loss.item():.3f})")
        del model, opt; torch.cuda.empty_cache()

    # --- 2. serve both adapters from ONE engine, in ONE batch ---
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    llm = LLM(model=BASE, enable_lora=True, max_lora_rank=8, max_loras=2,
              dtype="half", max_model_len=1024, gpu_memory_utilization=0.85)

    prompt = tok.apply_chat_template([{"role": "user", "content": "Describe a storm at sea."}],
                                     add_generation_prompt=True, tokenize=False)
    sp = SamplingParams(temperature=0.0, max_tokens=60)

    print("\n--- base model (no adapter) ---")
    print(llm.generate([prompt], sp)[0].outputs[0].text.strip()[:200])
    for i, name in enumerate(STYLES, start=1):
        out = llm.generate([prompt], sp, lora_request=LoRARequest(name, i, f"/content/adapter_{name}"))
        print(f"\n--- adapter '{name}' ---")
        print(out[0].outputs[0].text.strip()[:200])

    print("\nOne base model in memory. Two tenants. Two personalities. Same GPU, same engine.")

## Part 6 · The economics

This is the slide that gets multi-LoRA approved:

In [ ]:
def tenancy_cost(n_tenants, gpu_hr=1.80, base_gb=16.1, adapter_mb=134,
                 gpu_mem_gb=40, tenant_rps=0.3, gpu_capacity_rps=15.0):
    '''Dedicated GPU per tenant vs multi-LoRA on shared GPUs.'''
    dedicated_gpus = n_tenants                                   # 1 each, mostly idle
    total_rps = n_tenants * tenant_rps
    shared_gpus = max(1, math.ceil(total_rps / gpu_capacity_rps))
    # sanity: do the adapters even fit alongside the base model?
    adapters_per_gpu = math.ceil(n_tenants / shared_gpus)
    mem_needed = base_gb + adapters_per_gpu * adapter_mb / 1000
    return {"dedicated": dedicated_gpus, "shared": shared_gpus,
            "dedicated_mo": dedicated_gpus * gpu_hr * 730,
            "shared_mo": shared_gpus * gpu_hr * 730,
            "mem_ok": mem_needed <= gpu_mem_gb, "mem_needed": mem_needed,
            "util_dedicated": tenant_rps / gpu_capacity_rps}

print(f"{'tenants':>8}{'dedicated GPUs':>16}{'multi-LoRA GPUs':>17}{'$/mo dedicated':>17}"
      f"{'$/mo multi-LoRA':>17}{'saving':>9}")
print("-" * 86)
for n in (5, 10, 25, 50, 100, 250):
    c = tenancy_cost(n)
    saving = 1 - c["shared_mo"] / c["dedicated_mo"]
    fit = "" if c["mem_ok"] else "  ⚠ adapters exceed VRAM"
    print(f"{n:>8}{c['dedicated']:>16}{c['shared']:>17}{c['dedicated_mo']:>16,.0f}$"
          f"{c['shared_mo']:>16,.0f}${saving:>8.0%}{fit}")

c = tenancy_cost(50)
print(f"\nAt 50 tenants each sending 0.3 req/s: dedicated GPUs would run at "
      f"{c['util_dedicated']:.0%} utilization each.")
print("You would be renting 50 GPUs to leave 49 of them idle. Multi-LoRA turns that into "
      f"{c['shared']} GPU(s).")
print("\nCaveats worth stating out loud to whoever approves this:")
print("  - noisy-neighbour risk: one tenant's burst affects everyone (rate-limit per tenant)")
print("  - blast radius: one engine crash takes down all tenants (replicas, distributed-serving)")
print("  - per-tenant SLOs need per-tenant metrics (label your dashboards by adapter, logs)")

## Recap — and the end of the road

1. **Adapters are megabytes; base models are gigabytes.** 50 tenants ≈ one base model + ~7 GB.
2. **Batched multi-LoRA keeps the expensive GEMM shared** and groups the tiny rank-r matmuls, so
   many tenants decode in the same batch for a few percent overhead.
3. **`max_loras` is a scheduling constraint**, not just a memory knob — too low and you serialize
   tenants exactly like static batching serialized requests (Part 3/4).
4. **The adapter is just the model name** in the request — tenancy becomes a routing detail.
5. **Economically it's not close**: dedicated GPUs sit idle; shared multi-LoRA turns 50 GPUs into 1–2.

### The complete serving track so far

| Notebook | The one thing |
|---|---|
| [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) | Decode is memory-bound; KV memory is the scarce resource |
| [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) | Paging + continuous batching = the modern engine |
| [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) | Fewer weight bytes = faster decode + more KV |
| [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) | Spend idle compute guessing, losslessly |
| [Serving Internals Visualized](./Serving_Internals_Visualized_D3.ipynb) | See the scheduler and block pool move |
| [Reading the Logs](./Serving_Logs_Observability.ipynb) | Alert on causes (KV, queue), not symptoms (latency) |
| [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) | Goodput and $/1M tokens decide everything |
| [Structured Output](./Structured_Output_Guided_Decoding.ipynb) | Make invalid output unrepresentable |
| [Distributed Serving](./Distributed_MultiReplica_Serving.ipynb) | Replicas for throughput, TP for fit; routing is free money |
| **this one** | One base model, many tenants, one GPU |

### Further reading
- [LoRA](https://arxiv.org/abs/2106.09685) · [S-LoRA](https://arxiv.org/abs/2311.03285) (thousands of concurrent adapters) · [Punica](https://arxiv.org/abs/2310.18547) (the SGMV kernels)
- [vLLM LoRA docs](https://docs.vllm.ai/en/latest/features/lora.html)
- Where the adapters come from: [[LoRA & QLoRA Fine-Tuning](./LoRA_QLoRA_FineTuning.ipynb) — LoRA/QLoRA fine-tuning](./LoRA_QLoRA_FineTuning.ipynb)

🏁 **That's the whole series: train it (1–20), then serve it (21–30).**
[Back to the learning path](README.md).